# 选修E2 · Day 2：客户生命周期分析：CLV与流失预测

> **真实库**：pandas + numpy + scikit-learn + causaldata
> **真实数据**：NSW (National Supported Work) 真实 RCT 数据，445 条真实样本
> **核心任务**：CLV 三方法 + RFM 分群 + BG/NBD 简化CLV + 流失预测 (LogReg/RF) + 行动矩阵

**学习目标**：
1. 用 pandas+numpy 实现历史CLV和简单预测CLV公式
2. 用 pandas 实现 RFM 五分群（Champions/Recent/At Risk/Hibernating/Lost）
3. 用 pandas+numpy 实现 BG/NBD 简化版 CLV 预测
4. 用 sklearn LogisticRegression 训练流失预测模型并评估 AUC/Precision/Recall
5. 用 sklearn RandomForest 对比并解读 classification_report
6. 用 CLV × 流失风险构建四象限行动矩阵 + 特征重要性

## 环境准备：导入真实库

**真实库说明**：
- pandas + numpy：CLV 公式与 RFM 分群基础
- sklearn：流失预测模型 (LogisticRegression / RandomForest) + 评估 (AUC-ROC / Precision / Recall)
- causaldata：NSW 真实 RCT 数据

In [ ]:
import pandas as pd
import numpy as np
from causaldata.nsw_mixtape import load_pandas
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (classification_report, roc_auc_score,
                             precision_score, recall_score, confusion_matrix)

print('库导入完成：pandas + numpy + sklearn + causaldata')
print(f'pandas version: {pd.__version__}')
print(f'numpy version: {np.__version__}')
import sklearn; print(f'sklearn version: {sklearn.__version__}')

## TODO1：历史 CLV + 简单预测 CLV

**营销场景**：CLV（Customer Lifetime Value）是营销分析最重要的预测指标。先计算客户的历史 CLV（已产生的总价值），再用月均消费 × 预期月数预测未来 CLV。

**映射逻辑**：
- 历史 CLV = re74 + re75 + re78（3 年总消费）
- 月均消费 = re78 / 12（用活动后年消费作为月均代理）
- 留存率 = (re78 > 0).mean()（活动后仍消费的比例）
- 预期月数 = 1 / 流失率 = 1 / (1 - 留存率)
- 简单预测 CLV = 月均消费 × 预期月数

**任务**：
1. 加载 NSW 数据
2. 计算 `hist_clv` = re74 + re75 + re78
3. 计算留存率与预期月数
4. 计算 `simple_pred_clv` = (re78/12) × 预期月数
5. 打印 describe() 与均值/中位数

In [ ]:
# TODO1: 历史 CLV + 简单预测 CLV
# 提示：hist_clv = re74+re75+re78, 留存率=(re78>0).mean(), 预期月数=1/(1-留存率)
# TODO: 你的代码
raise NotImplementedError

# 1. 加载 NSW 数据
# df = load_pandas().data.copy()

# 2. 计算历史 CLV = re74 + re75 + re78

# 3. 计算留存率与预期月数

# 4. 计算简单预测 CLV = (re78/12) × 预期月数

# 5. 打印 describe 与均值/中位数

## TODO2：RFM 客户分群

**营销场景**：用 RFM（Recency / Frequency / Monetary）将客户分为 5 个可行动分群，指导差异化营销策略。

**映射逻辑**：
- **R**ecency（最近性）：re78 > 0 = 1（近期有消费），re78 = 0 = 0（近期无消费）
- **F**requency（频次）：3 年中消费年份计数 (re74>0) + (re75>0) + (re78>0)
- **M**onetary（金额）：re78（活动后消费金额）

**分群规则**（5 类）：
| R | F | 分群 |
|---|---|------|
| 1 | >=2 | Champions（冠军：近期+多频） |
| 1 | 1 | Recent（近期：近期+低频） |
| 0 | >=2 | At Risk（风险：无近期+多频） |
| 0 | 1 | Hibernating（休眠：无近期+低频） |
| 0 | 0 | Lost（流失：完全无消费） |

**任务**：
1. 计算 R, F, M 三个维度
2. 用 `pd.qcut` 对 M 分四分位（M_score）
3. 用 apply + 自定义函数分群
4. 打印 `segment` value_counts

In [ ]:
# TODO2: RFM 客户分群
# 提示：R=(re78>0), F=(re74>0)+(re75>0)+(re78>0), M=re78
# TODO: 你的代码
raise NotImplementedError

# 1. 计算 R, F, M

# 2. 对 M 分四分位 M_score

# 3. 定义分群函数并 apply

# 4. 打印 segment value_counts

## TODO3：BG/NBD 简化版 CLV 预测

**营销场景**：用 BG/NBD（Beta Geometric / Negative Binomial Distribution）的简化版公式预测客户未来 365 天 CLV，并按价值四分位分群。

**BG/NBD 核心思想**（Peter Fader & Bruce Hardie 2005）：
- 客户在活跃期间以一定频率购买（Poisson 过程）
- 客户在每次购买后有一定概率流失（Beta 分布）
- 两个行为独立假设

**简化公式**（避免 `lifetimes` 库依赖冲突，专注模型思想）：
```
avg_order_value = hist_clv / F  # 客户平均订单金额
bg_nbd_clv = F * (retention_rate^12) * avg_order_value * 12 * discount_factor
              |__________|      |_____|          |_|     |_|
              未来购买次数     月数(12)        贴现(0.99)
```

**任务**：
1. 计算 `avg_order_value` = hist_clv / F（F=0 时用 1 避免除零）
2. 用上述公式计算 `bg_nbd_clv`
3. 用 `pd.qcut` 将 CLV 分为 4 分位（低/中低/中高/高价值）
4. 打印 CLV describe + value_seg value_counts

In [ ]:
# TODO3: BG/NBD 简化版 CLV 预测
# 提示：AOV = hist_clv / F.replace(0,1); bg_nbd_clv = F * (retention**12) * AOV * 12 * 0.99
# TODO: 你的代码
raise NotImplementedError

# 1. 计算 avg_order_value

# 2. 计算 bg_nbd_clv（用 retention_rate 来自 TODO1）

# 3. 用 pd.qcut 分四分位（低/中低/中高/高价值）

# 4. 打印 CLV describe + value_seg value_counts

## TODO4：流失标签构造 + LogisticRegression 训练

**营销场景**：预测哪些客户会流失（活动后无消费），用 LogisticRegression 作为基线模型。

**流失定义**：re78 == 0（活动后无消费 = 流失）

**特征工程**（仅用活动前/基线特征，避免泄漏）：
- treat：是否收到营销干预（核心特征）
- age, educ, marr, nodegree：人口统计学协变量
- re74, re75：历史消费基线
- baseline_freq：3年中消费年份数 (re74>0)+(re75>0)
- baseline_aov：基线平均订单金额 = (re74+re75)/baseline_freq
- engagement_score：age*0.3 + educ*2（消费能力代理）
- tenure：age - 18（成人期长度）

**任务**：
1. 构造 `churn` 标签 = (re78 == 0).astype(int)
2. 计算上述所有特征
3. 用 `train_test_split(stratify=y, test_size=0.2, random_state=42)` 分割
4. 用 `StandardScaler` 标准化训练集与测试集
5. 训练 `LogisticRegression(class_weight='balanced', max_iter=2000)`
6. 计算 AUC-ROC, Precision, Recall

In [ ]:
# TODO4: 流失标签 + LogisticRegression
# 提示：churn=(re78==0); StandardScaler 标准化; class_weight='balanced'
# TODO: 你的代码
raise NotImplementedError

# 1. 构造 churn 标签

# 2. 特征工程（baseline_freq, baseline_aov, engagement_score, tenure）

# 3. 准备 X, y 并 train_test_split(stratify=y)

# 4. StandardScaler 标准化

# 5. 训练 LogisticRegression(class_weight='balanced', max_iter=2000)

# 6. 计算 AUC-ROC, Precision, Recall

## TODO5：RandomForest 对比 + classification_report

**营销场景**：用 RandomForest 作为非线性强模型与 LogisticRegression 基线对比，输出完整 classification_report 和特征重要性。

**任务**：
1. 训练 `RandomForestClassifier(n_estimators=300, max_depth=6, class_weight='balanced', random_state=42)`
2. 预测并计算 AUC-ROC, Precision, Recall
3. 打印 `classification_report`（完整报告）
4. 与 TODO4 的 LogisticRegression 对比（哪个 AUC 更高？）
5. 计算特征重要性并排序（注意：RF 不需要标准化，直接用原始 X_train）

**业务解读关键**：
- AUC-ROC > 0.80 是工业级可用门槛
- 不平衡场景下看 Precision/Recall 比 Accuracy 更有意义
- Precision = 预测流失中真流失的比例（误挽留成本）
- Recall = 真流失中被预测出的比例（漏挽留成本）

In [ ]:
# TODO5: RandomForest 对比 + classification_report
# 提示：RF 不需标准化; n_estimators=300, max_depth=6, class_weight='balanced'
# TODO: 你的代码
raise NotImplementedError

# 1. 训练 RandomForest（直接用 X_train, 无需标准化）

# 2. 预测并计算 AUC-ROC, Precision, Recall

# 3. 打印 classification_report

# 4. 与 LogReg 对比

# 5. 计算特征重要性并排序

## TODO6：CLV × 流失风险四象限行动矩阵

**营销场景**：将 CLV 预测（TODO3）与流失概率（TODO4/5）组合为四象限行动矩阵，为每个象限设计差异化营销行动。这是预测性分析升级到处方性分析的关键一步。

**四象限**：
| | 高 CLV | 低 CLV |
|---|---|---|
| **高流失风险** | Q1: 优先挽留（专属客户经理/深度折扣） | Q3: 低成本挽留（自动化邮件/Push） |
| **低流失风险** | Q2: 价值提升（向上销售/交叉销售） | Q4: 维持现状（标准服务） |

**任务**：
1. 对全量数据预测流失概率（用 RF 模型）
2. 用 CLV 中位数和流失概率中位数作为分界
3. 用 apply 函数将每个客户分入 Q1/Q2/Q3/Q4
4. 打印四象限 value_counts
5. 计算并打印 Q1 客户的总 CLV（优先挽留可挽回的价值）
6. 打印最终特征重要性（完整版）

In [ ]:
# TODO6: CLV × 流失风险四象限行动矩阵
# 提示：churn_prob=rf.predict_proba(X)[:,1]; CLV=df['bg_nbd_clv']; 用中位数作为分界
# TODO: 你的代码
raise NotImplementedError

# 1. 对全量数据预测流失概率

# 2. 用 CLV 中位数和流失概率中位数作为分界

# 3. 用 apply 将每个客户分入 Q1/Q2/Q3/Q4

# 4. 打印四象限 value_counts

# 5. 计算 Q1 客户的总 CLV

# 6. 打印完整特征重要性

## 总结：CLV + 流失预测 -> 处方性营销行动

完成 6 个 TODO 后，你应能回答：

1. **CLV 评估**：历史 CLV 与简单预测 CLV 差距多大？BG/NBD 简化 CLV 与简单预测 CLV 哪个更保守？
2. **RFM 分群**：Champions 有多少？At Risk 有多少？M_score 四分位的金额阈值是多少？
3. **流失预测**：LogisticRegression 和 RandomForest 的 AUC 各是多少？为什么 AUC 接近 0.5？（提示：RCT 随机化设计）
4. **业务行动**：Q1（高 CLV 高风险）有多少客户？可挽回的总价值多少？

**关键认知**：
- CLV 将营销决策从'短期 ROI'转向'长期客户价值'
- 流失预测的真正价值不在模型本身，而在与 CLV 组合形成行动矩阵
- AUC ~0.54 不是模型失败，而是 RCT 设计的体现--真实营销需要更丰富的行为特征
- Precision/Recall 比 Accuracy 更适合不平衡场景

**下一步（Day 3）**：将 CLV 与营销组合优化（MMM/MTA）结合，形成完整的'预测 + 处方'营销分析闭环。